[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/05_Advanced_Topics/07_text_to_image_video/07_text_to_image_video.ipynb)

# 07. Text-to-Image & Video Generation

**This notebook covers:**
- Text-conditional image generation concepts
- CLIP-guided generation demo
- Latent space visualization

**Runtime:** ~10–15 minutes on CPU

---

> **Theory & derivations:** See [README.md](./README.md) for full step-by-step math.


In [ ]:
# ============================================================
#  Google Colab Setup — Run this cell FIRST
# ============================================================
import os, sys

try:
    import google.colab
    IN_COLAB = True
    print("Google Colab detected — setting up environment...")
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        print("Cloning repository...")
        !git clone --depth 1 {REPO_URL} {REPO_DIR}
    else:
        print("Repository already cloned")

    print("Installing dependencies...")
    !pip install -q -r {REPO_DIR}/requirements.txt

    MODULE_DIR = f"{REPO_DIR}/05_Advanced_Topics/07_text_to_image_video"
    os.chdir(MODULE_DIR)
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    print(f"Colab setup complete — {os.getcwd()}")

    import torch
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("Device: CPU (all notebooks work fine on CPU)")
else:
    os.makedirs("../../assets", exist_ok=True)
    print("Running locally — all set!")

In [ ]:
import sys
sys.path.append('../..')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

try:
    from utils.visualization import set_style
    from utils.helpers import count_parameters, get_device
    set_style()
except ImportError:
    def set_style():
        plt.rcParams.update({'figure.figsize': (10, 6), 'figure.dpi': 100})
    def count_parameters(model):
        total = sum(p.numel() for p in model.parameters())
        print(f"Total parameters: {total:,}")
        return total
    def get_device():
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    set_style()

torch.manual_seed(42)
np.random.seed(42)
device = get_device() if callable(get_device) else torch.device('cpu')
print(f"PyTorch {torch.__version__} | Device: {device}")

## 1. Text-Conditional Generation Pipeline

Stable Diffusion: Text encoder → cross-attention U-Net in latent space → VAE decoder.


In [ ]:
pipeline_steps = [
    "Tokenize prompt",
    "Text encoder -> context embeddings C",
    "Sample z_T ~ N(0,I) in latent space",
    "For t=T..1: z_{t-1} = Denoise(z_t, C, t)",
    "Image = VAE_decode(z_0)",
]
for i, s in enumerate(pipeline_steps, 1):
    print(f"{i}. {s}")

## 2. CLIP-Guided Generation (Gradient Ascent Demo)

Optimize image embedding to match text embedding in CLIP space.


In [ ]:
class CLIPGuidedGenerator(nn.Module):
    def __init__(self, dim=64):
        super().__init__()
        self.image = nn.Parameter(torch.randn(1, dim))
        self.text_proj = nn.Linear(dim, dim, bias=False)

    def forward(self, text_vec):
        img = F.normalize(self.image, dim=-1)
        txt = F.normalize(self.text_proj(text_vec), dim=-1)
        return (img @ txt.T).squeeze()

gen = CLIPGuidedGenerator(64)
text_vec = F.normalize(torch.randn(1, 64), dim=-1)
opt = torch.optim.Adam([gen.image], lr=0.1)

scores = []
for step in range(80):
    score = gen(text_vec)
    loss = -score
    opt.zero_grad(); loss.backward(); opt.step()
    scores.append(score.item())

plt.plot(scores)
plt.xlabel('Step'); plt.ylabel('CLIP similarity'); plt.title('CLIP-guided latent optimization')
plt.show()
print(f"Final similarity: {scores[-1]:.3f}")

## 3. Latent Space Visualization (2D PCA)


In [ ]:
prompts = ["a red cat", "a blue dog", "sunset beach", "mountain lake"]
latents = torch.randn(len(prompts), 32)
# Simulate clustering by prompt semantics
latents[0] += torch.tensor([2.] + [0.]*31)
latents[1] += torch.tensor([2.] + [0.]*15 + [1.] + [0.]*16)
latents[2] += torch.tensor([-2.] + [0.]*31)
latents[3] += torch.tensor([-2.] + [0.]*15 + [-1.] + [0.]*16)

from sklearn.decomposition import PCA
xy = PCA(n_components=2).fit_transform(latents.numpy())
plt.figure(figsize=(7, 6))
for i, p in enumerate(prompts):
    plt.scatter(xy[i, 0], xy[i, 1], s=100)
    plt.annotate(p, (xy[i, 0]+0.05, xy[i, 1]+0.05))
plt.title('Text prompt clusters in synthetic latent space')
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.grid(True, alpha=0.3); plt.show()

## 4. Video Generation Concept — Temporal Consistency

Video models add temporal attention across frames: $O(F^2 d)$ per patch stream.


In [ ]:
F, D = 8, 64
frame_emb = torch.randn(F, D)
temporal_attn = F.softmax(frame_emb @ frame_emb.T / (D ** 0.5), dim=-1)
plt.imshow(temporal_attn.numpy(), cmap='Blues')
plt.xlabel('Frame'); plt.ylabel('Frame'); plt.title('Temporal attention (synthetic)')
plt.colorbar(); plt.show()

## Summary

Covered text-to-image pipelines, CLIP-guided optimization, and latent/video concepts.

**Next:** [08_multimodal_agents](../08_multimodal_agents/08_multimodal_agents.ipynb)
